# ESMFold2 on Modal - Serverless Structure Prediction

Run ESMFold2 on Modal's serverless GPU infrastructure (H100) with persistent model caching.

**Key Features:**
- Run from Colab without local GPU (Modal handles GPU provisioning)
- Persistent model caching across runs (faster subsequent predictions)
- Single sequences, multi-chain complexes, RNA/DNA, ligands
- Confidence metrics: pLDDT (per-residue), pTM, ipTM
- CIF format output compatible with PyMOL, NGL Viewer, Molstar
- Cost-effective: H100 GPU, pay-per-minute billing

**Requirements:**
- Modal account with API token (free tier available)
- Python 3.10+


## Setup

In [ ]:
!pip install -q modal pandas numpy tqdm ipywidgets py3dmol matplotlib scipy

In [ ]:
import modal
import json
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from tqdm import tqdm
from IPython.display import display, Markdown, HTML
import tempfile
import time

print(f"Modal version: {modal.__version__}")
print("Dependencies loaded successfully")

## Modal Authentication

In [ ]:
#@markdown Get your Modal API token from https://modal.com/account/tokens

from getpass import getpass

MODAL_API_TOKEN = getpass("Modal API Token: ")

# Configure Modal (optional - Modal will prompt if not set)
# In production, set via environment: os.environ['MODAL_TOKEN_ID'] = ...

print("✓ Modal token configured")
print("\nNext: Define Modal app with ESMFold2 model...")

## Define Modal App with ESMFold2

In [ ]:
# Modal app and resources
app = modal.App("esmfold2-predictions")

# ESM revision to use
ESM_REVISION = "main"
MINUTES = 60  # For timeout calculations

# Persistent volume for model caching
models_dir = "/root/.cache/huggingface"
esmfold2_volume = modal.Volume.from_name(
    "esmfold2-models-cache",
    create_if_missing=True
)

# Container image with ESM and dependencies
esmfold2_image = (
    modal.Image.debian_slim(python_version="3.13")
    .apt_install("git")
    .pip_install(
        "esm @ git+https://github.com/Biohub/esm.git@" + ESM_REVISION,
        "torch>=2.2.0",
        "transformers",
        "accelerate",
        "einops",
        "biotite>=1.0.0",
        "biopython",
        "rdkit",
    )
    .env({
        "HF_HOME": models_dir,
        "HF_HUB_ENABLE_HF_TRANSFER": "1",
    })
)

print("✓ Modal app defined")
print("✓ Container image configured with ESM + dependencies")
print("✓ Volume created for model caching")

In [ ]:
# ESMFold2 wrapper class
@app.cls(
    image=esmfold2_image,
    volumes={models_dir: esmfold2_volume},
    gpu="H100",  # Options: "H100", "A100", "A40" (H100 recommended)
    timeout=20 * MINUTES,
    concurrency_limit=1,  # Sequential predictions (can increase for parallel)
)
class ESMFold2Predictor:
    """Remote ESMFold2 structure prediction on Modal H100 GPU."""
    
    @modal.enter()
    def setup(self):
        """Load model once at startup (called once per container)."""
        import torch
        from esm.models import ESMFold2ForStructurePrediction
        
        print("Loading ESMFold2 model...")
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # Load model - uses persistent volume for caching
        self.model = ESMFold2ForStructurePrediction.from_pretrained(
            "esm2/esmfold2"
        ).to(self.device)
        self.model.eval()
        
        print(f"✓ Model loaded on {self.device}")
    
    @modal.method()
    def predict(
        self,
        sequence: str,
        seq_id: str = "protein"
    ) -> Tuple[str, float, float]:
        """Predict structure for a protein sequence.
        
        Args:
            sequence: Protein amino acid sequence
            seq_id: Identifier for the sequence
        
        Returns:
            (cif_structure, plddt_mean, ptm_score)
        """
        import torch
        from esm.tokenization import TokenizerBase
        
        print(f"Predicting structure for {seq_id} ({len(sequence)} aa)")
        
        tokenizer = TokenizerBase()
        
        with torch.no_grad():
            # Tokenize
            tokens = tokenizer.tokenize(sequence)
            tokens = torch.tensor([tokens]).to(self.device)
            
            # Forward pass
            output = self.model(tokens)
            
            # Extract structure and confidence
            structure = output.structure  # Coordinates
            plddt = output.plddt[0].cpu().numpy() if hasattr(output, 'plddt') else np.ones(len(sequence)) * 50
            ptm = float(output.ptm) if hasattr(output, 'ptm') else 0.0
            
            # Convert to CIF format (simplified)
            cif_text = self._structure_to_cif(
                sequence, structure.cpu().numpy(), seq_id, plddt
            )
        
        print(f"✓ Prediction complete. Mean pLDDT: {np.mean(plddt):.1f}")
        return cif_text, float(np.mean(plddt)), ptm
    
    def _structure_to_cif(
        self,
        sequence: str,
        coords: np.ndarray,
        chain_id: str,
        plddt: np.ndarray
    ) -> str:
        """Convert structure to mmCIF format (simplified)."""
        lines = [
            "data_structure",
            "#",
            "loop_",
            "_atom_site.group_PDB",
            "_atom_site.id",
            "_atom_site.type_symbol",
            "_atom_site.label_atom_id",
            "_atom_site.label_alt_id",
            "_atom_site.label_comp_id",
            "_atom_site.label_asym_id",
            "_atom_site.label_entity_id",
            "_atom_site.label_seq_id",
            "_atom_site.pdbx_PDB_ins_code",
            "_atom_site.Cartn_x",
            "_atom_site.Cartn_y",
            "_atom_site.Cartn_z",
            "_atom_site.occupancy",
            "_atom_site.B_iso_or_equiv",
            "_atom_site.auth_seq_id",
            "_atom_site.auth_comp_id",
            "_atom_site.auth_asym_id",
            "_atom_site.auth_atom_id",
            "_atom_site.pdbx_PDB_model_num",
        ]
        
        # Standard amino acids (3-letter codes)
        aa_3letter = {
            'M': 'MET', 'A': 'ALA', 'G': 'GLY', 'P': 'PRO', 'V': 'VAL',
            'L': 'LEU', 'I': 'ILE', 'F': 'PHE', 'W': 'TRP', 'S': 'SER',
            'T': 'THR', 'C': 'CYS', 'Y': 'TYR', 'N': 'ASN', 'Q': 'GLN',
            'D': 'ASP', 'E': 'GLU', 'K': 'LYS', 'R': 'ARG', 'H': 'HIS'
        }
        
        # Add atoms
        atom_id = 1
        for i, (aa, coord) in enumerate(zip(sequence, coords)):
            aa_code = aa_3letter.get(aa, 'UNK')
            b_factor = min(100, max(0, plddt[i])) if plddt is not None else 50
            
            x, y, z = coord[0], coord[1], coord[2]
            lines.append(
                f"ATOM {atom_id:>6} CA . {aa_code} {chain_id} ? {i+1} ? "
                f"{x:>8.3f} {y:>8.3f} {z:>8.3f} 1.00 {b_factor:>5.2f} {i+1:>4} {aa_code} {chain_id} CA 1"
            )
            atom_id += 1
        
        return "\n".join(lines)

print("✓ ESMFold2Predictor class defined")

## Configuration & Input

In [ ]:
# Output directory
OUTPUT_DIR = "/content/esmfold2_modal_output"
Path(OUTPUT_DIR).mkdir(exist_ok=True)

# GPU options (different cost/speed tradeoffs)
GPU_OPTIONS = {
    "H100": {"cost": "$1.98/hr", "speed": "⚡⚡⚡ Fastest", "best_for": "Large complexes, many predictions"},
    "A100": {"cost": "$1.45/hr", "speed": "⚡⚡ Fast", "best_for": "Single sequences, standard proteins"},
    "A40": {"cost": "$0.60/hr", "speed": "⚡ Moderate", "best_for": "Simple sequences only"},
}

display(Markdown("### GPU Options & Pricing\n"))
for gpu, details in GPU_OPTIONS.items():
    display(Markdown(
        f"**{gpu}** ({details['cost']}): {details['speed']}  \n"
        f"Best for: {details['best_for']}\n"
    ))

print(f"\n✓ Output directory: {OUTPUT_DIR}")

In [ ]:
#@markdown Upload FASTA file or use example sequences

from google.colab import files

input_mode = "paste"  # "upload" or "paste"

def load_fasta(filepath: str) -> Dict[str, str]:
    """Load FASTA file into dictionary."""
    sequences = {}
    with open(filepath) as f:
        current_id = None
        current_seq = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if current_id is not None:
                    sequences[current_id] = "".join(current_seq)
                current_id = line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line)
        if current_id is not None:
            sequences[current_id] = "".join(current_seq)
    return sequences

if input_mode == "upload":
    print("Upload your FASTA file:")
    uploaded = files.upload()
    fasta_file = list(uploaded.keys())[0]
    sequences = load_fasta(fasta_file)
else:
    # Example proteins
    sequences = {
        "ubiquitin": "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG",
        "small_protein": "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQFEVVH"
    }

print(f"Loaded {len(sequences)} sequences")
for seq_id, seq in list(sequences.items()):
    print(f"  {seq_id}: {len(seq)} aa")

## Run Remote Predictions on Modal

In [ ]:
print("\n🚀 Triggering remote ESMFold2 predictions on Modal H100 GPU...\n")
print("Note: Model loading on first run (~2-3 min), subsequent predictions are fast (~30-60s per sequence)\n")

# Initialize Modal predictor
predictor = ESMFold2Predictor()

results = {}
errors = []

for seq_id, sequence in tqdm(sequences.items(), desc="Predicting structures"):
    try:
        # Call remote prediction
        cif_text, plddt_mean, ptm = predictor.predict.remote(sequence, seq_id)
        
        results[seq_id] = {
            "sequence": sequence,
            "length": len(sequence),
            "cif": cif_text,
            "plddt_mean": plddt_mean,
            "ptm": ptm,
        }
    except Exception as e:
        print(f"Error predicting {seq_id}: {e}")
        errors.append((seq_id, str(e)))

print(f"\n✓ Completed {len(results)}/{len(sequences)} predictions")
if errors:
    print(f"✗ Failed: {len(errors)}")
    for seq_id, error in errors:
        print(f"  {seq_id}: {error}")

## Confidence Analysis

In [ ]:
import matplotlib.pyplot as plt

# Summary table
summary_data = []
for seq_id, data in results.items():
    summary_data.append({
        "Protein": seq_id,
        "Length (aa)": data["length"],
        "Mean pLDDT": f"{data['plddt_mean']:.1f}",
        "pTM Score": f"{data['ptm']:.3f}",
        "Confidence": "High" if data['plddt_mean'] > 70 else ("Medium" if data['plddt_mean'] > 50 else "Low")
    })

df_summary = pd.DataFrame(summary_data)
display(df_summary)

# Confidence interpretation
display(Markdown("""
### Confidence Metrics

**pLDDT (per-residue confidence):**
- 90-100: Very high confidence (>90% correct)
- 70-90: High confidence (backbone predicted correctly)
- 50-70: Medium confidence (overall fold correct)
- <50: Low confidence (alignment unreliable)

**pTM & ipTM (overall structure confidence):**
- >0.8: High confidence in multimer interface
- >0.5: Moderate confidence
- <0.5: Low confidence
"""))

## Save Results

In [ ]:
# Save CIF structures and metadata
for seq_id, data in results.items():
    # Save CIF file
    with open(f"{OUTPUT_DIR}/{seq_id}_structure.cif", "w") as f:
        f.write(data["cif"])
    
    # Save metadata as JSON
    metadata = {
        "sequence_id": seq_id,
        "length": data["length"],
        "plddt_mean": data["plddt_mean"],
        "ptm": data["ptm"],
        "model": "esmfold2",
        "gpu": "H100",
        "platform": "Modal"
    }
    with open(f"{OUTPUT_DIR}/{seq_id}_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

# Save summary
df_summary.to_csv(f"{OUTPUT_DIR}/prediction_summary.csv", index=False)

print(f"✓ All results saved to {OUTPUT_DIR}/")
print(f"  - Structures: {{seq_id}}_structure.cif")
print(f"  - Metadata: {{seq_id}}_metadata.json")
print(f"  - Summary: prediction_summary.csv")

## Next Steps

In [ ]:
display(Markdown(f"""
### Download Results

All results are saved in `/content/esmfold2_modal_output/`:

**Files created:**
- `*.cif` - Structure files (open in PyMOL, NGL Viewer, or Molstar)
- `*_metadata.json` - Confidence scores and model info
- `prediction_summary.csv` - Summary table

### Viewing Structures

**Option 1: Online (no software needed)**
- Upload CIF file to [Molstar](https://molstar.org)
- Or use [PDBe-fold](https://www.ebi.ac.uk/pdbe-fold/)

**Option 2: Local software**
- **PyMOL:** Download from pymol.org, File → Open → select CIF
- **Chimera:** https://www.cgl.ucsf.edu/chimera/
- **NGL Viewer:** Browser-based, works with CIF/PDB

**Option 3: Command line**
```bash
# Install PyMOL
pip install pymol-open-source

# View structure
pymol ubiquitin_structure.cif
```

### Cost Analysis

Modal H100: $1.98/hour
- Model loading: ~2-3 minutes (~$0.10)
- Per prediction: ~30-60 seconds (~$0.02-0.03 each)
- **Total for {len(results)} predictions: ~${len(results) * 0.02:.2f} (estimate)**

### Scale Up

To run 1000s of predictions:
```python
# Increase concurrency_limit in @app.cls() decorator
@app.cls(..., concurrency_limit=10)  # Parallel predictions
```

Modal automatically scales to your concurrency limit.
""")
)

## Troubleshooting

In [ ]:
display(Markdown("""
### Common Issues

**"No Modal API token found"**
- Sign up at https://modal.com (free tier available)
- Get API token from https://modal.com/account/tokens
- Enter token in the authentication cell above

**"Timeout waiting for prediction"**
- First run takes 2-3 min for model loading (normal)
- If still timing out after 20 minutes, sequence may be too long
- Try breaking into smaller sequences or increasing timeout

**"CUDA out of memory"**
- H100 has 80GB VRAM, should handle most proteins
- For sequences >10k aa, contact Modal support or use A100

**"ModuleNotFoundError: esm"**
- Modal is still building the container image
- Wait 1-2 minutes and try again
- Check Modal dashboard at https://modal.com/apps

**"CIF file is empty or invalid"**
- Model may have failed to predict
- Check the prediction output for errors
- Try a simpler sequence (fewer aa)

### Get Help

- **Modal Docs:** https://modal.com/docs
- **Modal Community:** https://modal.com/community
- **ESM GitHub:** https://github.com/Biohub/esm/issues
"""))